### 1. Installing Requirements
First, we install the necessary libraries: `datasets` for downloading data, `evaluate` for computing metrics, and `transformers` for the models.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

### 2. Preparing the Data
We load the MRPC dataset, instantiate our tokenizer, and map it over the dataset to create `input_ids`. We also create a `DataCollatorWithPadding` which dynamically pads batches during training so that all sequences in a batch have the same length.

In [18]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

### 3. Setting up TrainingArguments
The `TrainingArguments` class contains all the hyperparameters the `Trainer` will use for training and evaluation. The only required argument is a directory where the model checkpoints will be saved (e.g., `"test-trainer"`).

In [19]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

### 4. Loading the Model
We load our pre-trained model but with a sequence classification head on top. We specify `num_labels=2` because our dataset has two classes (equivalent or not equivalent). You will see a warning because the pre-trained weights of the classification head are randomly initialized—this is exactly what we are going to train (fine-tune) now!

In [20]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 5. Instantiating the Trainer (Without Evaluation)
We pass our model, arguments, datasets, collator, and tokenizer to the `Trainer`. By default, the `Trainer` handles the entire training loop for us. Note that we haven't provided a way to evaluate the model yet, so it will only compute the training loss.

In [21]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

### 6. Starting the Training (First pass)
Calling `trainer.train()` starts the fine-tuning process. The weights of the classification head are being adjusted. However, because we didn't specify evaluation metrics, we only see the training loss dropping, which doesn't tell us how well the model generalizes.

In [22]:
trainer.train()

Step,Training Loss
500,0.500793
1000,0.251389


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.3019421396691815, metrics={'train_runtime': 223.9302, 'train_samples_per_second': 49.14, 'train_steps_per_second': 6.149, 'total_flos': 405114969714960.0, 'train_loss': 0.3019421396691815, 'epoch': 3.0})

### 7. Getting Predictions
To see how our model is actually doing, we can use `trainer.predict()` on the validation set. This returns an object containing the raw logits (predictions), the actual labels, and some metrics.

In [23]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


### 8. Processing Predictions
The predictions are raw logits of shape `(408, 2)`. To compare them to the labels, we take the index of the highest logit for each prediction using `np.argmax(..., axis=-1)`.

In [24]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

### 9. Computing Metrics
We use the `evaluate` library to load the metric associated with the MRPC dataset. We then pass our processed predictions and the true labels to compute our accuracy and F1 score.

In [25]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8455882352941176, 'f1': 0.8923076923076924}

### 10. Automating Evaluation in the Trainer
Instead of evaluating manually after training, we can wrap our evaluation logic into a function called `compute_metrics`. The `Trainer` will call this function during training to automatically evaluate the model.

In [26]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

### 11. Creating a New Trainer with Evaluation
We create a new `TrainingArguments` object, this time specifying `eval_strategy="epoch"`. This tells the `Trainer` to evaluate the model at the end of each epoch. We also instantiate a *new* model (to start from scratch) and pass our `compute_metrics` function to the `Trainer`.

In [28]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 12. Final Fine-Tuning with Evaluation
We run `trainer.train()` again. This time, you will see the Validation Loss, Accuracy, and F1 score printed out after every epoch. This tells us exactly how our fine-tuning is improving the model's performance on unseen data!

In [29]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.415609,0.821078,0.876897
2,0.516850,0.509112,0.840686,0.891122
3,0.278974,0.721623,0.843137,0.892617


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.3322240764365823, metrics={'train_runtime': 250.2744, 'train_samples_per_second': 43.968, 'train_steps_per_second': 5.502, 'total_flos': 405114969714960.0, 'train_loss': 0.3322240764365823, 'epoch': 3.0})